In [6]:
from config import ROOT_DIR

import pandas as pd

In [7]:
# 두 군집화 결과 불러오기 및 병합
kmeans_df = pd.read_csv(ROOT_DIR / 'data/KMeans_clusters.csv')
dbscan_df = pd.read_csv(ROOT_DIR / 'data/DBSCAN_clusters.csv')

df = pd.merge(kmeans_df, dbscan_df[['CustomerID', 'DBSCAN_Cluster']], on='CustomerID', how='inner')

In [8]:
features = ['PurchaseCount', 'TotalSpent', 'AvgUnitPrice', 'UniqueItems']

In [9]:
# KMeans 군집별 통계
kmeans_summary = df.groupby('Cluster')[features].agg(['mean', 'count'])
kmeans_summary

PurchaseCount           TotalSpent       AvgUnitPrice        \
                 mean count           mean count         mean count   
Cluster                                                               
0            2.614907  3703     905.285675  3703     4.132855  3703   
1           12.270665   617    6009.425462   617     3.175898   617   
2           75.117647    17  108781.555294    17     4.976971    17   
3            1.000000     1    2033.100000     1  2033.100000     1   

        UniqueItems        
               mean count  
Cluster                    
0         38.395625  3703  
1        182.675851   617  
2        700.058824    17  
3          1.000000     1

# 전체 고객을 균등하게 군집화

Cluster 0: 다수 고객(3,700명 이상)으로, 구매 횟수, 총 구매액, 구매 품목 수가 상대적으로 낮고 단가가 보통인 그룹

Cluster 1: 중간 규모(600명 정도)로 구매 횟수와 총 구매액, 품목 수가 꽤 높은 편이나 단가는 낮은 편

Cluster 2: 소수(17명)지만 매우 충성도가 높고, 구매액과 품목 수가 매우 큰 VIP 고객군

Cluster 3: 단 한 명으로 특이치 같으며, 평균 단가가 매우 높은 1회성 구매

In [10]:
# DBSCAN 군집별 통계 (노이즈는 제외하지 않고 포함)
dbscan_summary = df.groupby('DBSCAN_Cluster')[features].agg(['mean', 'count'])
dbscan_summary

PurchaseCount          TotalSpent       AvgUnitPrice        \
                        mean count          mean count         mean count   
DBSCAN_Cluster                                                              
-1                 28.937500   128  28418.611641   128    40.357891   128   
 0                  3.504884  4197   1248.115879  4197     3.296997  4197   
 1                 15.142857     7   4510.350000     7     2.197571     7   
 2                  2.000000     6    651.803333     6    60.418333     6   

               UniqueItems        
                      mean count  
DBSCAN_Cluster                    
-1              250.343750   128  
 0               55.249702  4197  
 1              406.571429     7  
 2                3.166667     6

# 이상치 탐지, 핵심 고객군만 집중 분석
Noise(-1): 구매 횟수, 총 구매액, 평균 단가, 품목 수가 모두 매우 높은 고객들도 일부 포함하는 이질적인 그룹

Cluster 0: 가장 많은 고객(4,197명)으로, 적당한 구매 횟수와 총 구매액, 평균 단가 및 품목 수를 보임

Cluster 1: 소수(7명)로 구매 횟수와 총 구매액, 구매 품목 수가 큰 편이나 평균 단가는 낮음

Cluster 2: 6명 소수로, 구매 횟수는 낮지만 평균 단가가 매우 높고 품목 수는 적은 특이한 고객군